# Word Vector Exercise

In [ ]:
import gensim.downloader as api

print("모델 로딩...")
model = api.load("glove-wiki-gigaword-50")
model

In [ ]:
print("1. 단어 간 유사도 계산")
word1, word2, word3 = "cat", "dog", "car"

print(model.similarity(word1, word2))
print(model.similarity(word1, word3))

In [ ]:
print("2. 비슷한 단어 가져오기")
words = model.most_similar("computer", topn=5)
for word, score in words:
    print(word, score)

In [ ]:
print("3. 가장 안어울리는 것 찾기")
group = ["apple", "banana", "orange", "car"]
print(model.doesnt_match(group))

# Dataset Exercise

In [ ]:
import importlib
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello, world. I am a student."
integers = tokenizer.encode(text)
print(integers)
print(tokenizer.decode(integers))

In [ ]:
import os
import requests

os.makedirs("datas", exist_ok=True)
url = (
    "https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
    "the-verdict.txt"
)
file_path = "datas/the-verdict.txt"
response = requests.get(url, timeout=30)
response.raise_for_status()
with open(file_path, "wb") as f:
    f.write(response.content)

with open("datas/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(raw_text[50:250])

In [ ]:
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

enc_sample = enc_text[50:]
print(len(enc_sample))

context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1 : context_size + 1]

print(x)
print(y)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDataset(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        print("token_ids 길이", len(token_ids))

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


import tiktoken


def create_dataloader_v1(
    txt,
    batch_size=4,
    max_length=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    num_workers=0,
):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDataset(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )
    return dataloader


dataloader = create_dataloader_v1(
    raw_text,
    batch_size=1,
    max_length=4,
    stride=1,
    shuffle=False,
)

data_iter = iter(dataloader)
print(next(data_iter))
print(next(data_iter))
print(next(data_iter))

In [ ]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(
    vocab_size, output_dim
)  # num_embeddings, embedding_dim
print(embedding_layer.weight)

input_ids = torch.tensor([2, 3, 5, 1])
print(embedding_layer(input_ids))

In [ ]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

max_length = 4
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=max_length,
    stride=max_length,
    shuffle=False,
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)  # batch_size(8), token_cnt(4), embedding_dim(256)

context_length = max_length

pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

input_embedding = (
    token_embeddings + pos_embeddings
)  # 각 Chunk들마다 pos_embeddings 덧셈.
print(input_embedding.shape)

In [ ]:
from transformers import AutoTokenizer

qwen = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
exaone = AutoTokenizer.from_pretrained("LGAI-EXAONE/EXAONE-4.0-1.2B")

texts = [
    "안녕하세요, 반갑습니다!",
    "Hello, nice to meet you!",
    "인공지능 기술이 발전하고 있습니다.",
    "Hello! This is an English tokenizer test.",
    "こんにちは！日本語のテストです。",
    "人工智能技术正在快速发展。",
    "The quick brown fox jumps over the lazy dog. 빠른 갈색 여우가 게으른 개를 뛰어넘습니다.",
    "파이썬 프로그래밍은 매우 유용합니다. Python programming is very useful.",
    "123456789 !@#$%^&*() 특수문자",
]

for text in texts:
    print("텍스트", text)

    qwen_tokens = qwen.encode(text, add_special_tokens=True)
    qwen_tokens_strs = [qwen.decode([tok]) for tok in qwen_tokens[:10]]
    print("Qwen", qwen_tokens_strs)

    exaone_tokens = exaone.encode(text, add_special_tokens=True)
    exaone_tokens_strs = [exaone.decode([tok]) for tok in exaone_tokens[:10]]
    print("Exaone", exaone_tokens_strs)

print()
print()

for token_name in ["bos_token", "eos_token", "unk_token", "pad_token"]:
    token = getattr(qwen, token_name, None)
    if token:
        print(token_name, token, getattr(qwen, f"{token_name}_id", None))
    else:
        print(token_name, "없음")

print()
print()

for token_name in ["bos_token", "eos_token", "unk_token", "pad_token"]:
    token = getattr(exaone, token_name, None)
    if token:
        print(token_name, token, getattr(exaone, f"{token_name}_id", None))
    else:
        print(token_name, "없음")

# Basic MLP

In [ ]:
import torch
import torch.nn as nn
import tiktoken

torch.manual_seed(123)

tokenizer = tiktoken.get_encoding("gpt2")
vocab_size = tokenizer.n_vocab
block_size = 8

train_loader = create_dataloader_v1(
    raw_text,
    batch_size=32,
    max_length=block_size,
    stride=1,
    shuffle=False,
)  # Batch 당 32개의 input, target chunk 쌍으로 구성됨.


class MLP(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=64, hidden=256):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.fc1 = nn.Linear(block_size * emb_dim, hidden)
        self.fc2 = nn.Linear(hidden, vocab_size)

    def forward(
        self, x
    ):  # x = 입력 2차원 벡터. (batch 내 chunk(block) 갯수, chunk(block) 내 token 갯수)
        e = self.emb(x)  # e = 토큰들을 임베딩값으로 치환함.
        h = e.flatten(
            1
        )  # (B(Chunk 갯수), T(토큰 갯수), C(임베딩 차원)) 에서 T와 C를 평탄화함.
        h = self.fc1(h)  # 선형함수 통과, 가중치 곱셈 후 합 (+bias)
        h = nn.functional.relu(h)  # 활성화 함수 통과.
        return self.fc2(h)  # 출력층 선형함수 통과


model = MLP(vocab_size, block_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

model.train()
for epoch in range(5):
    total = 0
    for input_batch, target_batch in train_loader:
        logits = model(input_batch)
        loss = nn.functional.cross_entropy(logits, target_batch[:, -1])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
    print(f"epoch {epoch} | loss 평균 {total / len(train_loader): .4f}")


@torch.no_grad()
def generate(model, prompt, max_new_token=30):
    model.eval()
    ids = tokenizer.encode(prompt)
    pad_id = tokenizer.encode(" ")[0]
    for _ in range(max_new_token):
        ctx = ids[-block_size:]
        ctx = [pad_id] * (block_size - len(ctx)) + ctx
        logits = model(torch.tensor([ctx]))
        probs = nn.functional.softmax(
            logits, dim=-1
        )  # logits은 배치사이즈(1) * vocab 사이즈이니, 마지막 vocab 사이즈의 벡터에서 소프트맥스.
        ids.append(torch.multinomial(probs, 1).item())  # probs 중 샘플링 1개
    return tokenizer.decode(ids)


print(generate(model, "I had always thought Jack Gisburn"))

In [ ]:
import torch
import torch.nn as nn
import tiktoken

torch.manual_seed(123)

tokenizer = tiktoken.get_encoding("gpt2")
vocab_size = tokenizer.n_vocab
block_size = 8

train_loader = create_dataloader_v1(
    raw_text,
    batch_size=32,
    max_length=block_size,
    stride=1,
    shuffle=True,
)


class RNN(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, hidden=256):
        super().__init__()
        self.hidden = hidden
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.W_xh = nn.Linear(emb_dim, hidden)
        self.W_hh = nn.Linear(
            hidden, hidden, bias=False
        )  # 이전 은닉 상태를 현재 계산에 가중치와 함께 계산해줌.
        self.fc = nn.Linear(hidden, vocab_size)  # 출력층

    def forward(self, x):
        B, T = x.shape  # 입력 x는 2차원 텐서. [배치사이즈, 블록(청크)사이즈]
        e = self.emb(
            x
        )  # 블록(청크) 내 토큰들을 64차원의 임베딩값으로 바꿈. e는 3차원 텐서. [배치사이즈, 블록(청크)사이즈, 임베딩 차원]
        h = torch.zeros(B, self.hidden)  # 2차원 텐서. [배치사이즈, 은닉층 노드 갯수]

        outs = []
        for t in range(T):
            x_t = e[:, t, :]  # 블록(청크)별 t번째 토큰의 임베딩값
            h = self.W_xh(x_t) + self.W_hh(
                h
            )  # (현재 토큰 * weight + bias) + (이전 은닉상태 * weight)
            h = torch.tanh(h)  # 활성화함수. 현재 은닉상태 계산.
            outs.append(
                h
            )  # 블록(청크) 별 t번째 토큰들의 은닉상태들이 누적됨. outs[0]에는 모든 Chunk들의 0번째 은닉상태 값들이 있음.

        H = torch.stack(outs, dim=1)
        return self.fc(H)


model = RNN(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

model.train()
for epoch in range(5):
    total = 0
    for input_batch, target_batch in train_loader:
        logits = model(
            input_batch
        )  # input_batch에는 여러개(32)의 블록(청크)가 있고, 청크별로 여러개(8)의 토큰이 있고,
        # 토큰별로 여러개(50257)의 확률값이 순전파를 진행하면 나온다.
        loss = nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
        # logits.flatten(0, 1)은 0축부터 1축까지 모두 Flatten 한다. (32*8, 50257) (블록순번이랑 토큰순서를 하나로, 그리고 각 토큰id별로 확률값이 존재.)
        # target_batch.flatten()은 모든 축을 Flatten한다. (32*8,) 블록순번이랑 토큰순서를 하나로, 그리고 토큰id가 존재. (정답표)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    print(f"epoch {epoch} | loss 평균 {total / len(train_loader): .4f}")


@torch.no_grad()
def generate(model, prompt, max_new_token=30):
    model.eval()
    ids = tokenizer.encode(prompt)
    for _ in range(max_new_token):
        ctx = ids[-block_size:]  # 토큰idx 중 뒤에서부터 블록(청크)사이즈만큼 추출
        logits = model(torch.tensor([ctx]))  # 배치 형식으로 넣어줌.
        # logits 결과는 (배치사이즈(1), 블록(청크)사이즈(8), vocab 사이즈) 그리고 각각에 점수값
        logits = logits[:, -1, :]  # 마지막 토큰의 결과만 추출. 차원(축) 축소됨
        probs = nn.functional.softmax(logits, dim=-1) # vocab 별 점수값들에 대해 softmax
        ids.append(torch.multinomial(probs, 1).item())
    return tokenizer.decode(ids)


print(generate(model, "I had always thought Jack Gisburn"))